# Open Tesla Reality View — v1 Training & Demo on Google Colab

**Stack:** `nvidia/Alpamayo2-Edge` (Frozen SigLIP2 Driving Vision Tower, 0.41B) + `Depth Anything 3` / 3D Ego-Car Anchor + `Wan-AI/Wan2.1-VACE-1.3B-diffusers` (Flow-Matching DiT)

Recommended Colab runtimes: **RTX PRO 6000 (96GB)**, **H100 (80GB)**, or **A100 (40/80GB)**.

In [ ]:
# 1. Clone repo & install dependencies
!git clone https://github.com/EnglandLobster/open-tesla-reality-view.git
%cd open-tesla-reality-view
!pip install -q -e ".[train,dev]"
!python scripts/check_env.py

In [ ]:
# 2. Verify unit tests & real-weight GPU smoke test
!pytest -v
!python scripts/smoke_real_weights.py --height 256 --width 448 --num-frames 9 --denoise-steps 2

In [ ]:
# 3. Stage 0: Pre-cache frozen Alpamayo2-Edge tokens + Wan2.1 VAE / VACE control latents
# Replace --synthetic-demo with --clips-dir /content/drive/MyDrive/reality_view_clips
# once NuRec (nvidia/PhysicalAI-Autonomous-Vehicles-NuRec) or CARLA clips are rendered.
!python scripts/precache_features.py \
    --synthetic-demo 8 \
    --num-frames 17 \
    --height 480 \
    --width 832 \
    --out-dir data/cached_shards

In [ ]:
# 4. Stage 1A: Warm up AlpamayoContextAdapter (1152 -> 4096) with Wan2.1-VACE frozen
!python scripts/train_v1.py \
    --stage 1a \
    --shards-dir data/cached_shards \
    --output-dir runs/v1_colab \
    --max-steps 500 \
    --grad-accum 4 \
    --log-every 10 \
    --save-every 250

In [ ]:
# 5. Stage 1B: Joint fine-tuning (LoRA r=64 + VACE control blocks; use --stage 1b_full on PRO 6000 96GB / H100)
!python scripts/train_v1.py \
    --stage 1b_lora \
    --shards-dir data/cached_shards \
    --resume-adapter runs/v1_colab/stage_1a/adapter_step000500.pt \
    --output-dir runs/v1_colab \
    --max-steps 2000 \
    --grad-accum 4 \
    --log-every 20 \
    --save-every 500

In [ ]:
# 6. Run end-to-end v1 inference (supports interactive orbit via --orbit-azimuth-deg)
!python scripts/infer_v1.py \
    --adapter-ckpt runs/v1_colab/stage_1b_lora/adapter_step002000.pt \
    --num-frames 17 \
    --height 480 \
    --width 832 \
    --steps 20 \
    --orbit-azimuth-deg 15.0 \
    --output-dir outputs/v1_preview